# greCy_Atlomy demo

This notebook loads the published Atlomy spaCy pipeline and runs it on a few Greek passages, showing per-token annotations, span categorisation, and dependency rendering.

Before running:

```bash
pip install -r requirements.txt
pip install /path/to/grc_atlomy_spancat-0.1.0.tar.gz
```

(Or load directly from a model directory; see Cell 1.)

In [ ]:
import spacy
from spacy import displacy

# After `pip install grc_atlomy_spancat-0.1.0.tar.gz`, load by name:
#   nlp = spacy.load("grc_atlomy_spancat")
# Or load directly from a built model directory:
PIPELINE_PATH = "grc_atlomy_spancat"
nlp = spacy.load(PIPELINE_PATH)

print(f"Loaded: {nlp.meta['name']} v{nlp.meta['version']} (spaCy {nlp.meta['spacy_version']})")
print(f"Pipeline components: {nlp.pipe_names}")
print(f"SpanCat labels: {list(nlp.get_pipe('spancat').labels)}")

## Sample texts

Two passages — one anatomical (the model's training domain), one non-anatomical (out-of-domain Greek).

In [ ]:
TEXT_ANATOMICAL = (
    "πρὸς δὲ τὸν καυλὸν τὸν τῆς κύστεως συνήρτηται τὸ αἰδοῖον, "
    "τὸ μὲν ἐξωτάτω τρῆμα συνερρωγὸς εἰς τὸ αὐτό, μικρὸν δʼ ὑποκάτω"
)
TEXT_LITERARY = (
    "Ἀνατομικὰς ἐγχειρήσεις ἔγραψα μὲν καὶ πρόσθεν, ἡνίκα τὸ πρῶτον ἀνῆλθον ἔναγχος εἰς Ῥώμην."
)
TEXT_ANATOMICAL

## Per-token annotations

Token text, lemma, POS, fine-grained tag, dependency label, and morphological features.

In [ ]:
doc = nlp(TEXT_ANATOMICAL)

import pandas as pd
rows = [
    {
        "TEXT": t.text,
        "LEMMA": t.lemma_,
        "POS": t.pos_,
        "TAG": t.tag_,
        "DEP": t.dep_,
        "MORPH": str(t.morph),
    }
    for t in doc
]
pd.DataFrame(rows)

## Span categorisation

Spans live under `doc.spans['sc']` (the spancat's `spans_key`). Threshold is read from the component config.

In [ ]:
spans = doc.spans.get("sc", [])
print(f"{len(spans)} spans (threshold={nlp.get_pipe('spancat').cfg['threshold']:.3f})\n")
for s in spans:
    print(f"  [{s.start_char:3d}:{s.end_char:<5d}] {s.label_:<24s} {s.text!r}")

In [ ]:
SPAN_COLORS = {
    "Body Part":             "#ffadad",
    "Topography":            "#ffd6a5",
    "Adjectives/Qualities":  "#fdffb6",
    "Medical":               "#caffbf",
    "Pathology":             "#9bf6ff",
    "Physiology":            "#a0c4ff",
    "Technical Appellation": "#bdb2ff",
    "Division":              "#ffc6ff",
    "Action Verbs":          "#fffffc",
    "Symmetry/Opposition":   "#cccccc",
}

displacy.render(doc, style="span", jupyter=True,
                options={"spans_key": "sc", "colors": SPAN_COLORS})

## Dependency tree

First sentence rendered with displaCy. Lemma annotation enabled.

In [ ]:
first_sent = next(iter(doc.sents)).as_doc()
displacy.render(first_sent, style="dep", jupyter=True,
                options={"compact": True, "add_lemma": True, "distance": 130})

## Out-of-domain check

Same pipeline on a non-anatomical literary passage. Lemmatiser still works; spancat correctly produces few or no spans because the input has no anatomical terminology.

In [ ]:
doc2 = nlp(TEXT_LITERARY)
for t in doc2:
    if t.lemma_ != t.text:
        print(f"  {t.text:14s} -> {t.lemma_}")
print(f"\nSpans: {len(doc2.spans.get('sc', []))}")
for s in doc2.spans.get("sc", []):
    print(f"  [{s.start_char}:{s.end_char}] {s.text!r} -> {s.label_}")